In [1]:
import numpy as np 
import os 
import pandas as pd 
from dataclasses import dataclass
from pathlib import Path 


In [3]:
#!pip install pandas 

In [10]:
@dataclass
class CreateRagEngineConfig:
    root_dir : Path 
    faiss_index : str
    source_path : str 
    local_data_file : str

In [3]:
os.chdir("c:\\Users\\RSR\\PYTHON\\LLMIssueTracker\src")

In [4]:
from LLMIssueTracker.Utils.Common import * 
from LLMIssueTracker.Constant import * 


In [15]:
class ConfigurationManager:
    def __init__(self,
                 config_path =CONFIG_FILE_PATH,
                 params_path =PARAMS_FILE_PATH
                 ):
        os.chdir(r"C:\Users\RSR\PYTHON\LLMIssueTracker")
        self.config=Read_Yaml(Path(config_path))
        self.params=Read_Yaml(Path(params_path))
        
        self.Create_RagEngine=self.config.Create_Rag_Engine
    def get_ragengine_config(self)->CreateRagEngineConfig:
        config=self.Create_RagEngine
        create_directories([config.root_dir])

        ragengineconfig=CreateRagEngineConfig(
            root_dir=config.root_dir,
            faiss_index=config.faiss_index,
            source_path= config.source_path,
            local_data_file=config.local_data_file
        )
        return ragengineconfig


In [30]:
import faiss
import pickle
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import ollama

In [37]:
import faiss
import pickle
import ollama
from sentence_transformers import SentenceTransformer

class CreateRagEngine:

    def __init__(self, config):
        self.config = config

        # Load Sentence Transformer model
        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

        # Load FAISS index
        self.index = faiss.read_index(self.config.faiss_index)

        # Load documents
        with open(self.config.source_path, "rb") as file:
            self.docs = pickle.load(file)

    def search_similar(self, issue):

        # Create embedding for the new issue
        query_vector = self.model.encode([issue])

        # Search Top 5 similar incidents
        distances, indices = self.index.search(query_vector, k=5)

        # Retrieve similar documents
        results = []

        for idx in indices[0]:
            if idx != -1:
                results.append(self.docs[idx])

        # Prepare context
        context = "\n\n".join(results)

        # Prompt for LLM
        prompt = f"""
You are an Oracle Production Support Expert.

Below are historical production incidents.

Historical Incidents:
{context}

New Incident:
{issue}

Based on the historical incidents, provide:

1. Root Cause
2. Resolution Steps
3. SQL Fix (if required)
4. Oracle Package/Object Impact
5. Prevention Steps

Give the answer in a professional format.
"""
        print("Total documents :", len(self.docs))
        print("Top matched indexes :", indices[0])
        print("Distances :", distances[0])

        print("\nMatched Incidents:\n")

        for i in indices[0]:
            print("="*80)
            print(self.docs[i])
            
        try:
            response = ollama.chat(
                model="llama3.2",
                messages=[
                    {
                        "role": "user",
                        "content": prompt
                    }
                ]
            )

            return response["message"]["content"]

        except Exception as e:
            print(f"Error: {e}")
            return None

In [29]:
#!pip install faiss-cpu
#!pip install sentence-transformers
#!pip install ollama

In [ ]:
class CreateRagEngine:
    def __init__(self,
                 config :CreateRagEngineConfig):
        self.config=config
        self.model=SentenceTransformer("Sentence-transformers/all-miniLM-L6-v2")
        self.index=faiss.read_index(self.config.faiss_index)
        with open (self.config.source_path,"rb") as file :
            self.docs=pickle.load(file)
        
        self.client = OpenAI(
            api_key=""
        )
    def search_similar(self,issue):

        query_vector = self.model.encode([issue])

        distances, indices = self.index.search(
            query_vector,
            k=5
        )

        results = []

        for idx in indices[0]:
            results.append(self.docs[idx])
        
        
        context = "\n\n".join(results)

        prompt = f"""
        You are Oracle Production Support Expert.

        Historical incidents:

        {context}

        New Issue:

        {issue}

        Suggest:
        1. Root Cause
        2. Resolution
        3. SQL Fix
        4. Oracle Package Impact
        """

        # response = self.client.chat.completions.create(
        #     model="gpt-5",
        #     messages=[
        #         {
        #             "role":"user",
        #             "content":prompt
        #         }
        #     ]
        # )
        # #import ollama

        print("Total documents :", len(self.docs))
        print("Top matched indexes :", indices[0])
        print("Distances :", distances[0])

        print("\nMatched Incidents:\n")

        for i in indices[0]:
            print("="*80)
            print(self.docs[i])

        response = ollama.chat(
            model="llama3.2",
            messages=[
                {"role": "user", "content": "How to fix ORA-01722?"}
            ]
        )

        print(response["message"]["content"])

        return response.choices[0].message.content
        
    

        

In [38]:
try:
    incident='ORA-01722: invalid number'
    config_manager = ConfigurationManager()

    config = config_manager.get_ragengine_config()

    enginesearch = CreateRagEngine(config)

    resutls = enginesearch.search_similar(incident)
    print(f"Similar Incidents -{resutls}")

except Exception as e:
    raise e

Reading YAML : Config\Config.yaml
[2026-07-10 17:57:25,730: INFO: Common: YAML file loaded successfully : Config\Config.yaml]


Reading YAML : params.yaml
[2026-07-10 17:57:25,759: INFO: Common: YAML file loaded successfully : params.yaml]
[2026-07-10 17:57:25,776: INFO: Common: Created directory : artifacts/Rag_Engine]
[2026-07-10 17:57:25,888: INFO: model: No device provided, using cpu]
[2026-07-10 17:57:26,576: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"]
[2026-07-10 17:57:26,603: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"]
[2026-07-10 17:57:26,949: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"]
[2026-07-10 17:57:26,974: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 516.66it/s]


[2026-07-10 17:57:30,611: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"]
[2026-07-10 17:57:30,943: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-07-10 17:57:31,244: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-07-10 17:57:31,556: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"]
[2026-07-10 17:57:31,861: INFO: _client: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-07-10 17:57:31,892: INFO: _client: HTTP Request: HEAD h

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.12it/s]


Total documents : 7
Top matched indexes : [0 4 6 5 3]
Distances : [0.3219943  0.74347687 0.75632155 0.9907877  1.2325556 ]

Matched Incidents:


             Issue: ORA-01722: invalid number
             Root Cause: DATAINESERT
             Resolution: ORA-01722: invalid number
             

             Issue: ORA-01403: No Data Found
             Root Cause: PRA_EHB.TEVA_CUST.TEVA_CUST_OUT
             Resolution: nan
             

             Issue: ORA-00001: Unique Constraint Violated
             Root Cause: PRA_EHB.LEO_CUST.DATA_INSERT
             Resolution: nan
             

             Issue: ORA-01422: Exact Fetch Returns More Than Requested Number of Rows
             Root Cause: PRA_EHB.GILEAD_CUST.DATA_INSERT
             Resolution: nan
             

             Issue: |RAN PKG_HUB_ENGINE.CREATE_FILES(2121122)|RAN GET_DATA_BY_INTERFACE(22111223)RAN CLIENT_TRANSFORMATIONS|RAN UPDATE_COUNTRY_TBL|RAN UPDATE_SITE_TBL|RAN INSERRT_COUNTRY_EVENT|RAN INSERT_SITE_STATUS_E

In [ ]:
from openai import OpenAI



In [35]:
#!pip install openai